# データセットクラスの自作（Custom Dataset）

---
## 目的
PyTorchにおけるデータセットクラス（`torch.utils.data.Dataset`）の自作方法について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time
import os
import csv

import torch
import torch.nn as nn
from torchvision.transforms import v2
import torchsummary

from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの準備

CIFAR-10の全画像をPNG形式で1枚ずつ保存したデータセットをダウンロードします．
このデータセットは，画像ファイル（`.png`）と，画像ファイル名とラベルの対応を記録したCSVファイル（`train_labels.csv` / `test_labels.csv`）から構成されています．

In [ ]:
import gdown

gdown.download('https://drive.google.com/uc?id=1MZKAvZcuxC-o6tzuWfg1Px7TUqTguiCi', 'cifar10_png.zip', quiet=True)
!unzip -q cifar10_png.zip

## データセットクラスの定義

`torch.utils.data.Dataset`を継承して，独自のデータセットクラス`MyCIFAR10`を作成します．`Dataset`を継承したクラスでは，以下の3つのメソッドを定義する必要があります．

**`__init__(self, ...)`**

データセットの初期化を行います．ここでは，ラベル情報が記録されたCSVファイル（`train_labels.csv`または`test_labels.csv`）を読み込み，画像ファイル名とラベルのリストをメンバ変数として保持します．この時点では，画像ファイルそのものはまだ読み込みません．

**`__len__(self)`**

データセットのサンプル数を返します．ここでは，`__init__`で読み込んだファイル名のリストの長さを返すように定義します．

**`__getitem__(self, item)`**

`item`番目のサンプル（画像とラベル）を返します．このメソッドは，`DataLoader`がデータを1つ取り出すたびに呼び出されます．
ここで初めて，対応するPNG画像ファイルを`PIL.Image.open()`で読み込みます．このように，実際に必要になったタイミングで画像を読み込む方式にすることで，データセット全体を事前にメモリ上に展開する必要がなくなり，大規模なデータセットでも扱いやすくなります．

また，`transform`が指定されている場合は，読み込んだ画像に対して前処理・Data Augmentationを適用してから返します．`transform`の使い方は，`existing_dataset.ipynb`や`augmentation.ipynb`で紹介した`torchvision.datasets.CIFAR10`などの既存のデータセットクラスと同様です．

In [ ]:
class MyCIFAR10(torch.utils.data.Dataset):
    def __init__(self, root, train=True, transform=None):
        super().__init__()

        self.transform = transform

        split = 'train' if train else 'test'
        self.image_dir = os.path.join(root, 'cifar10_png', split)
        label_file = os.path.join(root, 'cifar10_png', f'{split}_labels.csv')

        # CSVファイルからファイル名とラベルの対応を読み込む
        self.filenames = []
        self.labels = []
        with open(label_file) as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.filenames.append(row['filename'])
                self.labels.append(int(row['label']))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, item):
        image_path = os.path.join(self.image_dir, self.filenames[item])
        image = Image.open(image_path).convert('RGB')
        label = self.labels[item]

        if self.transform is not None:
            image = self.transform(image)

        return image, label

## transformの定義とデータセットの読み込み

`torchvision.transforms.v2`を用いて，学習データにはランダムクロップと左右反転によるData Augmentationを，評価データには前処理のみを適用するように定義します．
定義した`transform`を`MyCIFAR10`の引数に渡すことで，データを読み込む際に自動的に適用されます．

In [ ]:
transform_train = v2.Compose([v2.RandomCrop(32, padding=4),
                               v2.RandomHorizontalFlip(),
                               v2.ToImage(),
                               v2.ToDtype(torch.float32, scale=True)])
transform_test = v2.Compose([v2.ToImage(),
                              v2.ToDtype(torch.float32, scale=True)])

train_data = MyCIFAR10(root="./", train=True, transform=transform_train)
test_data = MyCIFAR10(root="./", train=False, transform=transform_test)

print(len(train_data))
print(len(test_data))

image, label = train_data[10]
print(image.shape, label)

## ネットワークモデルの定義
畳み込みニューラルネットワークを定義します．ここでは，畳み込み層2層，全結合層3層から構成されるネットワークを使用します．

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.l1 = nn.Linear(8 * 8 * 32, 1024)
        self.l2 = nn.Linear(1024, 1024)
        self.l3 = nn.Linear(1024, 10)

    def forward(self, x):
        h = self.pool(self.relu(self.conv1(x)))
        h = self.pool(self.relu(self.conv2(h)))
        h = h.view(h.size()[0], -1)
        h = self.relu(self.l1(h))
        h = self.relu(self.l2(h))
        h = self.l3(h)
        return h

## ネットワークの作成
上で定義したネットワークを作成します．`.to(device)`を用いて，ネットワークを`device`で指定したデバイスへ配置します．

In [ ]:
model = CNN()
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
CIFAR-10データセットと作成したネットワークを用いて学習を行います．

In [ ]:
batch_size = 100
epoch_num = 10

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num+1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)
        loss = criterion(y, label)

        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(epoch, sum_loss/len(train_loader), count.item()/len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## 課題

1. `MyCIFAR10`の`__getitem__`を変更し，画像とラベルに加えて画像のファイル名も返すように変更してみましょう．